In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
DATA_DIR = "../data/external/Global-copper-deposit-dataset/"

porphyry = pd.read_csv(DATA_DIR + "USGS_MRDS_Porphyry_copper_deposit.csv", encoding="latin1")
sediment = pd.read_csv(DATA_DIR + "USGS_MRDS_Sed_copper_deposit.csv", encoding="latin1")
vms = pd.read_csv(DATA_DIR + "USGS_MRDS_VMS_deposit.csv", encoding="latin1")

porphyry["deposit_type"] = "Porphyry"
sediment["deposit_type"] = "Sediment"
vms["deposit_type"] = "VMS"

print(f"Porphyry: {porphyry.shape}")
print(f"Sediment: {sediment.shape}")
print(f"VMS:      {vms.shape}")

## Dataset unificado
Combinamos los 3 datasets usando solo las columnas comunes.

In [ ]:
common_cols = sorted(
    set(porphyry.columns) & set(sediment.columns) & set(vms.columns)
)
# Excluir deposit_type de comunes (la agregamos nosotros)
common_cols = [c for c in common_cols if c != "deposit_type"]
print(f"Columnas comunes ({len(common_cols)}):", common_cols)

df = pd.concat(
    [porphyry[common_cols + ["deposit_type"]],
     sediment[common_cols + ["deposit_type"]],
     vms[common_cols + ["deposit_type"]]],
    ignore_index=True
)
print(f"\nDataset combinado: {df.shape}")
df.head()

## Resumen general y valores nulos

In [ ]:
null_summary = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "porcentaje": (df.isnull().sum() / len(df) * 100).round(1),
    "dtype": df.dtypes
}).sort_values("porcentaje", ascending=False)

null_summary

## Variable objetivo: ley de cobre (`cugrd`)

In [ ]:
# Filtramos registros que tienen ley de cobre
df_cu = df.dropna(subset=["cugrd"]).copy()
print(f"Registros con cugrd: {len(df_cu)} / {len(df)} ({100*len(df_cu)/len(df):.1f}%)")
print()
print(df_cu["cugrd"].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histograma general
axes[0].hist(df_cu["cugrd"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribución de ley de cobre")
axes[0].set_xlabel("cugrd (%)")
axes[0].set_ylabel("Frecuencia")

# Histograma por tipo de depósito
for src in df_cu["deposit_type"].unique():
    subset = df_cu[df_cu["deposit_type"] == src]
    axes[1].hist(subset["cugrd"], bins=30, alpha=0.6, label=src, edgecolor="black")
axes[1].set_title("Ley de cobre por tipo de depósito")
axes[1].set_xlabel("cugrd (%)")
axes[1].legend()

# Boxplot por tipo
df_cu.boxplot(column="cugrd", by="deposit_type", ax=axes[2])
axes[2].set_title("Boxplot cugrd por tipo")
axes[2].set_xlabel("")
plt.suptitle("")

plt.tight_layout()
plt.show()

## Distribución geográfica

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Mapa de todos los depósitos
for src in df_cu["deposit_type"].unique():
    subset = df_cu[df_cu["deposit_type"] == src]
    axes[0].scatter(subset["longitude"], subset["latitude"], 
                    s=10, alpha=0.5, label=src)
axes[0].set_title("Ubicación de depósitos por tipo")
axes[0].set_xlabel("Longitud")
axes[0].set_ylabel("Latitud")
axes[0].legend()

# Mapa coloreado por ley de cobre
scatter = axes[1].scatter(df_cu["longitude"], df_cu["latitude"],
                          c=df_cu["cugrd"], cmap="YlOrRd", s=10, alpha=0.6)
axes[1].set_title("Depósitos coloreados por ley de cobre (%)")
axes[1].set_xlabel("Longitud")
axes[1].set_ylabel("Latitud")
plt.colorbar(scatter, ax=axes[1], label="cugrd (%)")

plt.tight_layout()
plt.show()

## Tonelaje vs ley de cobre

In [ ]:
df_ton = df_cu.dropna(subset=["oreton"]).copy()
print(f"Registros con cugrd Y oreton: {len(df_ton)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Escala normal
for src in df_ton["deposit_type"].unique():
    subset = df_ton[df_ton["deposit_type"] == src]
    axes[0].scatter(subset["oreton"], subset["cugrd"], s=15, alpha=0.5, label=src)
axes[0].set_xlabel("Tonelaje (oreton)")
axes[0].set_ylabel("Ley de cobre (%)")
axes[0].set_title("Tonelaje vs Ley de Cobre")
axes[0].legend()

# Escala log en tonelaje
for src in df_ton["deposit_type"].unique():
    subset = df_ton[df_ton["deposit_type"] == src]
    axes[1].scatter(subset["oreton"], subset["cugrd"], s=15, alpha=0.5, label=src)
axes[1].set_xscale("log")
axes[1].set_xlabel("Tonelaje (log)")
axes[1].set_ylabel("Ley de cobre (%)")
axes[1].set_title("Tonelaje vs Ley de Cobre (escala log)")
axes[1].legend()

plt.tight_layout()
plt.show()

## Correlaciones numéricas

In [ ]:
num_cols = df_cu.select_dtypes(include=[np.number]).columns.tolist()
corr = df_cu[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

## Depósitos por país (top 15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Conteo por país
top_countries = df_cu["country"].value_counts().head(15)
top_countries.plot(kind="barh", ax=axes[0], edgecolor="black")
axes[0].set_title("Top 15 países por cantidad de depósitos")
axes[0].set_xlabel("Cantidad")
axes[0].invert_yaxis()

# Ley promedio por país (top 15 por conteo)
avg_cu = df_cu[df_cu["country"].isin(top_countries.index)].groupby("country")["cugrd"].mean().reindex(top_countries.index)
avg_cu.plot(kind="barh", ax=axes[1], color="coral", edgecolor="black")
axes[1].set_title("Ley de cobre promedio por país")
axes[1].set_xlabel("cugrd promedio (%)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Domaining: ley de cobre por tipo de depósito y subtipo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Por deposit_type (Porphyry, Sediment, VMS)
sns.violinplot(data=df_cu, x="deposit_type", y="cugrd", ax=axes[0], inner="quartile")
axes[0].set_title("Distribución de ley por tipo de depósito")
axes[0].set_ylabel("cugrd (%)")

# Por deptype (subtipo geológico)
sns.boxplot(data=df_cu, x="deptype", y="cugrd", ax=axes[1])
axes[1].set_title("Ley por subtipo geológico (deptype)")
axes[1].set_ylabel("cugrd (%)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print("\nEstadísticas por dominio (deposit_type):")
print(df_cu.groupby("deposit_type")["cugrd"].describe().round(3))

## Resumen del EDA

Hallazgos clave para documentar después de ejecutar:

In [ ]:
print("=== RESUMEN DEL EDA ===")
print(f"Total registros: {len(df)}")
print(f"Registros con ley de cobre: {len(df_cu)} ({100*len(df_cu)/len(df):.1f}%)")
print(f"Tipos de depósito: {df_cu['deposit_type'].unique().tolist()}")
print(f"Subtipos (deptype): {df_cu['deptype'].dropna().unique().tolist()}")
print(f"Países representados: {df_cu['country'].nunique()}")
print(f"\nLey de cobre (cugrd):")
print(f"  Media:   {df_cu['cugrd'].mean():.3f}%")
print(f"  Mediana: {df_cu['cugrd'].median():.3f}%")
print(f"  Std:     {df_cu['cugrd'].std():.3f}%")
print(f"  Min:     {df_cu['cugrd'].min():.3f}%")
print(f"  Max:     {df_cu['cugrd'].max():.3f}%")